# Klasifikasi Sentimen Komentar - Koperasi Desa Merah Putih (KDMP)
Notebook ini melakukan **(1) pembersihan data** dan **(2) klasifikasi sentimen** (`positif`, `netral`, `negatif`) dari komentar TikTok menggunakan **TF-IDF + model machine learning**.

**Alur:** Audit data -> Cleaning -> EDA -> Split train/test -> Bandingkan model -> Tuning -> Evaluasi -> Simpan model & prediksi teks baru.

## 1. Setup & Konfigurasi

In [ ]:
# Jalankan sekali jika library belum terpasang (hapus tanda # di baris berikut)
# !pip install -q pandas numpy scikit-learn matplotlib seaborn joblib Sastrawi emoji

In [ ]:
import warnings, re, os, joblib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
# ============== KONFIGURASI ==============
FILE_PATH       = "KDMP_berlabel_perbaikan.csv"   # lokasi file dataset
RANDOM_STATE    = 42
SAMPLE_N        = None    # isi angka (mis. 20000) untuk uji cepat; None = pakai seluruh data
USE_STEMMING    = False   # stemming Sastrawi: bisa menyatukan kata berimbuhan, tapi LAMBAT (~15-30 menit untuk data penuh; hasil di-cache)
CLEAN_CSV_PATH  = "KDMP_bersih.csv"
STEM_CACHE_PATH = "stem_cache.pkl"

## 2. Memuat & Mengaudit Data Mentah
Sebelum dibersihkan, kita periksa dulu masalah apa saja yang ada di dataset. Temuan utama pada data ini:
- Baris **duplikat** (`comment_id` ganda) - ada salinan komentar level 0 dengan kolom `level` kosong.
- Komentar **kosong** dan bernilai `#NAME?` (error spreadsheet).
- `parent_comment_id` rusak karena tersimpan dalam notasi ilmiah (`7.64e+18`), sehingga tidak dipakai sebagai fitur.
- Teks penuh **noise**: tag `[Sticker]`, `\r\n` literal, emoji, huruf berulang, slang, mention, URL.

In [ ]:
# Baca semua kolom sebagai string agar ID besar (19 digit) tidak rusak / berubah jadi notasi ilmiah
df_raw = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=True)
print("Ukuran data mentah :", df_raw.shape)
df_raw.head()

In [ ]:
print("=== Nilai kosong per kolom ===")
print(df_raw.isna().sum(), "\n")

print("=== Distribusi label (mentah) ===")
print(df_raw["sentimen"].value_counts(dropna=False), "\n")

print("=== Distribusi kolom level ===")
print(df_raw["level"].value_counts(dropna=False), "\n")

print("Baris duplikat penuh          :", df_raw.duplicated().sum())
print("comment_id duplikat           :", df_raw["comment_id"].duplicated().sum())
print("Komentar kosong/NaN           :", (df_raw["comment"].isna() | (df_raw["comment"].str.strip() == "")).sum())
print("Komentar bernilai '#NAME?'    :", (df_raw["comment"] == "#NAME?").sum())
print("Komentar berisi tag [Sticker] :", df_raw["comment"].str.contains(r"\[[Ss]ti(c)?ker\]", na=False).sum())
print("Komentar berisi baris baru (\\r\\n):", df_raw["comment"].str.contains(r"[\r\n]", na=False).sum())
print("Contoh parent_comment_id (rusak, notasi ilmiah):", df_raw["parent_comment_id"].dropna().unique()[:3])

## 3. Pembersihan Struktur Data
Label dirapikan, komentar kosong/error dibuang, duplikat dihapus, tipe data diperbaiki.

In [ ]:
log = []   # catatan jumlah baris tiap tahap pembersihan
def catat(tahap, d):
    log.append({"tahap": tahap, "jumlah_baris": len(d)})

df = df_raw.copy()
catat("Data mentah", df)

# 1) Rapikan label
df["sentimen"] = df["sentimen"].str.strip().str.lower()
df = df[df["sentimen"].isin(["positif", "netral", "negatif"])]
catat("Label valid (positif/netral/negatif)", df)

# 2) Buang teks kosong & error spreadsheet ('#NAME?')
df["comment"] = df["comment"].replace({"#NAME?": np.nan, "#N/A": np.nan, "#VALUE!": np.nan})
df["comment"] = df["comment"].str.strip()
df = df[df["comment"].notna() & (df["comment"] != "")]
catat("Komentar tidak kosong", df)

# 3) Hapus duplikat comment_id (file berisi salinan ganda dari komentar level 0)
df = df.drop_duplicates(subset="comment_id", keep="first")
catat("Hapus duplikat comment_id", df)

# 4) Konversi tipe data
df["create_time"] = pd.to_datetime(df["create_time"], errors="coerce")
df = df.dropna(subset=["create_time"])
df["level"] = pd.to_numeric(df["level"], errors="coerce")
# Level: kolom level rusak pada salinan ganda -> tentukan ulang dari parent_comment_id
df["is_reply"] = df["parent_comment_id"].notna().astype(int)
catat("Format tanggal valid", df)

# 5) Kolom ID/parent tidak dipakai sebagai fitur (ID hanya pengenal)
df = df.reset_index(drop=True)

if SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)
    catat(f"Sampling {SAMPLE_N} baris", df)

pd.DataFrame(log)

## 4. Pembersihan Teks (Preprocessing NLP)
Tahapan: *lowercase* -> hapus URL/mention/tag stiker/`\r\n` -> normalisasi huruf berulang -> **emoji diubah jadi token teks** (emoji membawa sentimen) -> hapus angka & simbol -> **normalisasi slang** -> hapus *stopword* (kata **negasi** seperti "tidak", "bukan" sengaja dipertahankan) -> *stemming* Sastrawi.

In [ ]:
import re, os, joblib, emoji
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# ---- Kamus slang / singkatan bahasa Indonesia -> bentuk baku ----
SLANG = {
    "gk":"tidak","ga":"tidak","gak":"tidak","nggak":"tidak","ngga":"tidak","enggak":"tidak","engga":"tidak",
    "tdk":"tidak","g":"tidak","tak":"tidak","kagak":"tidak","gpp":"tidak apa","gapapa":"tidak apa",
    "yg":"yang","yng":"yang","dgn":"dengan","dg":"dengan","dr":"dari","dri":"dari","utk":"untuk","untk":"untuk",
    "krn":"karena","karna":"karena","krna":"karena","sm":"sama","sama2":"sama","tp":"tapi","tapi":"tapi",
    "jd":"jadi","jdi":"jadi","udh":"sudah","udah":"sudah","sdh":"sudah","dah":"sudah","blm":"belum","belom":"belum",
    "bgt":"banget","bngt":"banget","bgd":"banget","skrg":"sekarang","skrang":"sekarang","sblm":"sebelum",
    "lg":"lagi","lgi":"lagi","lgsg":"langsung","kalo":"kalau","klo":"kalau","kl":"kalau","klw":"kalau",
    "emg":"memang","emang":"memang","mmg":"memang","hrs":"harus","harusny":"harus","bs":"bisa","bsa":"bisa",
    "sy":"saya","aq":"aku","ak":"aku","gw":"aku","gue":"aku","gua":"aku","w":"aku","lu":"kamu","loe":"kamu",
    "km":"kamu","kmu":"kamu","org":"orang","orng":"orang","spt":"seperti","sprti":"seperti","kyk":"seperti",
    "kayak":"seperti","bkn":"bukan","bukannya":"bukan","brp":"berapa","brapa":"berapa",
    "dmn":"dimana","dmna":"dimana","kmn":"kemana","knp":"kenapa","kenapa":"kenapa","gmn":"bagaimana","gimana":"bagaimana",
    "gmna":"bagaimana","bgmn":"bagaimana","emangnya":"memang","pd":"pada","pdhl":"padahal","pdahal":"padahal",
    "hrg":"harga","hrga":"harga","rb":"ribu","rbu":"ribu","jt":"juta","tmn":"teman","tmen":"teman",
    "mksh":"terima kasih","makasih":"terima kasih","thx":"terima kasih","tks":"terima kasih","trims":"terima kasih",
    "mantul":"mantap","mantab":"mantap","mantaap":"mantap","sip":"bagus","oke":"bagus","ok":"bagus","okey":"bagus",
    "bagu":"bagus","top":"bagus","keren":"bagus","joss":"bagus","sukses":"sukses","smoga":"semoga","semoga":"semoga",
    "moga":"semoga","amin":"amin","aamiin":"amin","amiin":"amin","btw":"omong omong","jgn":"jangan","jangn":"jangan",
    "sbg":"sebagai","sbgai":"sebagai","tsb":"tersebut","dll":"dan lain lain","dsb":"dan sebagainya","spy":"supaya",
    "biar":"supaya","cm":"cuma","cuman":"cuma","doang":"saja","aja":"saja","aj":"saja","sj":"saja","saja":"saja",
    "trus":"terus","trs":"terus","terus2":"terus","bnyk":"banyak","banyakk":"banyak","sdikit":"sedikit","skit":"sedikit",
    "lbh":"lebih","lbih":"lebih","mgkn":"mungkin","mungkin":"mungkin","tau":"tahu","tw":"tahu","pke":"pakai","pake":"pakai",
    "bikin":"buat","bkin":"buat","liat":"lihat","lht":"lihat","dpt":"dapat","dapet":"dapat","dpat":"dapat","ttp":"tetap",
    "tetep":"tetap","bru":"baru","baru2":"baru","lama2":"lama","cmn":"cuma","sll":"selalu","slalu":"selalu",
    "pemerintah":"pemerintah","pmrntah":"pemerintah","pemrintah":"pemerintah","pmerintah":"pemerintah",
    "indo":"indonesia","ri":"indonesia","wkwk":"tertawa","wkwkwk":"tertawa","wkwkwkwk":"tertawa","haha":"tertawa",
    "hahaha":"tertawa","hehe":"tertawa","hihi":"tertawa","kwkw":"tertawa","xixi":"tertawa","awokawok":"tertawa",
    "anjir":"kaget","anjay":"kaget","njir":"kaget","astaga":"kaget","astagfirullah":"istighfar",
}

# ---- Stopword: Sastrawi + tambahan; kata negasi DIPERTAHANKAN karena penting untuk sentimen ----
NEGASI = {"tidak","bukan","belum","jangan","tanpa","kurang","tak","nggak","gak","ga"}
STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
STOPWORDS |= {"nya","sih","deh","dong","lah","kah","pun","tuh","nih","ya","yah","kan","kok","lho","loh","mah","nah","wah",
              "mas","bang","bg","kak","kakak","sis","bro","min","pak","bapak","bu","ibu","om","tante","mba","mbak","mbk",
              "eh","oh","ah","hehe","hmm","aja","saja","ada","adalah","sih","tertawa","lho","gitu","gini","begitu","begini",
              "banget","sekali","sangat","juga","lagi","udah","sudah","aku","kamu","saya","kita","kami","mereka","dia","nya",
              "kalau","jadi","buat","apa","sama","semua","dulu","terus","tahu","dapat","biar","gitu","display_"}
STOPWORDS -= NEGASI
STOPWORDS -= {"tidak","bukan","belum","jangan","tanpa","kurang"}

# ---- Kata topik yang muncul di hampir semua komentar (tidak membedakan apa-apa) ----
DOMAIN_STOPWORDS = {"koperasi","koprasi","kopdes","kdmp","kmp","merah","putih","desa","kelurahan"}
STOPWORDS |= DOMAIN_STOPWORDS

In [ ]:
URL_RE      = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE  = re.compile(r"@[\w\.]+")
BRACKET_RE  = re.compile(r"\[[^\]]{1,25}\]")            # [Sticker], [laughwithtears], dll.
SKIN_RE     = re.compile(r"_(medium_light|medium_dark|light|medium|dark)_skin_tone")
REPEAT_RE   = re.compile(r"(.)\1{2,}")                   # 3+ karakter/emoji berulang -> 1

stemmer = StemmerFactory().create_stemmer() if USE_STEMMING else None
stem_cache = joblib.load(STEM_CACHE_PATH) if (USE_STEMMING and os.path.exists(STEM_CACHE_PATH)) else {}

NO_STEM = {"pemerintah", "presiden", "rakyat", "menteri"}   # Sastrawi keliru menstem kata ini (mis. pemerintah -> perintah)

def stem_word(w):
    if w in NO_STEM:
        return w
    if w not in stem_cache:
        stem_cache[w] = stemmer.stem(w)
    return stem_cache[w]

def clean_text(text):
    t = str(text)
    t = re.sub(r"\\r\\n|\\n|\\r", " ", t)                 # \r\n yang tertulis literal
    t = re.sub(r"[\r\n\t]+", " ", t)                      # newline sungguhan
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = BRACKET_RE.sub(" ", t)
    t = t.replace("#", " ")
    t = REPEAT_RE.sub(r"\1", t)                           # 'mantaaaap' -> 'mantap', 😂😂😂 -> 😂
    t = emoji.demojize(t, delimiters=(" emo_", " "))      # emoji -> token teks (tetap membawa sentimen)
    t = t.lower()
    t = SKIN_RE.sub("", t)
    t = re.sub(r"[^a-z_\s]", " ", t)                      # buang angka, tanda baca, simbol
    tokens = t.split()

    out = []
    for w in tokens:
        for x in SLANG.get(w, w).split():
            if x.startswith("emo_"):
                out.append(x)
                continue
            if x in STOPWORDS or len(x) < 2:
                continue
            if USE_STEMMING:
                x = stem_word(x)
            if x in STOPWORDS or len(x) < 2:
                continue
            out.append(x)
    # hilangkan token identik yang berurutan
    out = [w for i, w in enumerate(out) if i == 0 or w != out[i-1]]
    return " ".join(out)

# Uji singkat
for contoh in ["Mantaaaap pak!!! 😂😂😂 [Sticker] https://tiktok.com/x @budi", "Gk masuk akal, harga gas 16rb?? Pemerintah bohong\r\nbanget"]:
    print(contoh, "\n  ->", clean_text(contoh))

In [ ]:
df["text_clean"] = df["comment"].apply(clean_text)
if USE_STEMMING:
    joblib.dump(stem_cache, STEM_CACHE_PATH)      # simpan cache agar run berikutnya cepat

df["n_token"] = df["text_clean"].str.split().str.len()
df[["comment", "text_clean", "sentimen"]].sample(8, random_state=RANDOM_STATE)

## 5. Filter Akhir & Simpan Data Bersih

In [ ]:
# Buang komentar yang kosong setelah dibersihkan (mis. hanya stiker / stopword)
df = df[df["n_token"] > 0]
catat("Teks tidak kosong setelah preprocessing", df)

# Hapus teks bersih yang identik (duplikat / spam). Ini juga mencegah kebocoran data train-test.
# Jika teks identik punya label berbeda -> ambigu, dibuang seluruhnya.
n_label = df.groupby("text_clean")["sentimen"].transform("nunique")
df = df[n_label == 1]
catat("Buang teks ambigu (label bertentangan)", df)
df = df.drop_duplicates(subset="text_clean", keep="first").reset_index(drop=True)
catat("Hapus teks duplikat", df)

df = df[["video_id", "comment_id", "username", "create_time", "level", "is_reply",
         "comment", "text_clean", "n_token", "sentimen"]]
df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Data bersih tersimpan: {CLEAN_CSV_PATH}  ({len(df):,} baris)")
pd.DataFrame(log)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.DataFrame(log).set_index("tahap")["jumlah_baris"].plot.barh(ax=ax[0], color="#4C72B0")
ax[0].invert_yaxis(); ax[0].set_title("Jumlah baris di setiap tahap pembersihan"); ax[0].set_xlabel("baris")

urutan = ["negatif", "netral", "positif"]
warna  = {"negatif": "#C44E52", "netral": "#8C8C8C", "positif": "#55A868"}
cnt = df["sentimen"].value_counts().reindex(urutan)
ax[1].bar(cnt.index, cnt.values, color=[warna[k] for k in cnt.index])
for i, v in enumerate(cnt.values):
    ax[1].text(i, v, f"{v:,}\n({v/cnt.sum():.1%})", ha="center", va="bottom")
ax[1].set_title("Distribusi label setelah dibersihkan"); ax[1].set_ylim(0, cnt.max()*1.2)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
sns.histplot(df["n_token"].clip(upper=40), bins=40, ax=ax[0], color="#4C72B0")
ax[0].set_title("Panjang komentar (jumlah token, dipotong di 40)")
top = pd.Series(" ".join(df["text_clean"]).split()).value_counts().head(20)[::-1]
top.plot.barh(ax=ax[1], color="#DD8452"); ax[1].set_title("20 kata paling sering")
plt.tight_layout(); plt.show()

## 6. Pembagian Data Latih & Uji
Pembagian **stratified** 80:20 agar proporsi kelas sama di train dan test. Data tidak seimbang (netral dominan), maka evaluasi utama memakai **F1-macro** (bukan hanya akurasi).

In [ ]:
from sklearn.model_selection import train_test_split

X = df["text_clean"]
y = df["sentimen"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
print("\nProporsi kelas (train):\n", y_train.value_counts(normalize=True).round(3))
print("\nProporsi kelas (test):\n", y_test.value_counts(normalize=True).round(3))

## 7. Ekstraksi Fitur TF-IDF
Unigram + bigram (mis. "tidak jelas", "harga mahal"). *Vectorizer* hanya di-`fit` pada data latih supaya tidak ada kebocoran data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100_000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)
print("Bentuk matriks TF-IDF train:", X_train_vec.shape)

## 8. Membandingkan Beberapa Model
Semua model memakai `class_weight='balanced'` (bila tersedia) untuk menangani ketidakseimbangan kelas. **DummyClassifier** dipakai sebagai *baseline* pembanding.

In [ ]:
import time
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

models = {
    "Baseline (kelas mayoritas)": DummyClassifier(strategy="most_frequent"),
    "Naive Bayes (Complement)"  : ComplementNB(alpha=0.3),
    "Logistic Regression"       : LogisticRegression(max_iter=1000, C=2, class_weight="balanced"),
    "Linear SVM"                : LinearSVC(C=0.5, class_weight="balanced"),
    "SGD (modified huber)"      : SGDClassifier(loss="modified_huber", alpha=1e-5, class_weight="balanced",
                                                max_iter=50, random_state=RANDOM_STATE),
}

hasil = []
for nama, m in models.items():
    t0 = time.time()
    m.fit(X_train_vec, y_train)
    pred = m.predict(X_test_vec)
    hasil.append({
        "Model": nama,
        "Akurasi": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test, pred, average="macro"),
        "F1 (weighted)": f1_score(y_test, pred, average="weighted"),
        "Waktu (dtk)": time.time() - t0,
    })

df_hasil = pd.DataFrame(hasil).sort_values("F1 (macro)", ascending=False).reset_index(drop=True)
df_hasil.style.format({c: "{:.4f}" for c in df_hasil.columns if c not in ["Model", "Waktu (dtk)"]} | {"Waktu (dtk)": "{:.1f}"}) \
       .background_gradient(subset=["F1 (macro)"], cmap="Greens")

In [ ]:
ax = df_hasil.set_index("Model")[["Akurasi", "F1 (macro)", "F1 (weighted)"]].plot.barh(figsize=(9, 4), width=0.75)
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_title("Perbandingan performa model pada data uji")
plt.tight_layout(); plt.show()

## 9. Tuning Hyperparameter (Cross-Validation)
`GridSearchCV` (3-fold, skor **F1-macro**) untuk Logistic Regression dan Linear SVM. TF-IDF ikut dimasukkan ke dalam *pipeline* sehingga di-`fit` ulang di tiap fold (bebas kebocoran).
> Set `RUN_TUNING = False` jika ingin melewati tahap ini agar lebih cepat.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

RUN_TUNING = True

def buat_pipeline(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(min_df=3, max_df=0.95, sublinear_tf=True, max_features=100_000)),
        ("clf", clf),
    ])

grid_umum = {"tfidf__ngram_range": [(1, 1), (1, 2)], "clf__C": [0.3, 1, 3]}
kandidat = {
    "Logistic Regression": buat_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Linear SVM": buat_pipeline(LinearSVC(class_weight="balanced")),
}

if RUN_TUNING:
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    hasil_tuning = {}
    for nama, pipe in kandidat.items():
        gs = GridSearchCV(pipe, grid_umum, scoring="f1_macro", cv=cv, n_jobs=-1, verbose=1)
        gs.fit(X_train, y_train)
        hasil_tuning[nama] = gs
        print(f"{nama:22s} | CV F1-macro = {gs.best_score_:.4f} | parameter terbaik: {gs.best_params_}")
    nama_terbaik = max(hasil_tuning, key=lambda k: hasil_tuning[k].best_score_)
    best_model = hasil_tuning[nama_terbaik].best_estimator_
else:
    nama_terbaik = "Linear SVM"
    best_model = buat_pipeline(LinearSVC(C=0.5, class_weight="balanced")).set_params(tfidf__ngram_range=(1, 2))
    best_model.fit(X_train, y_train)

print("\nModel terpilih:", nama_terbaik)

## 10. Evaluasi Akhir pada Data Uji

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

labels = ["negatif", "netral", "positif"]
y_pred = best_model.predict(X_test)

print(f"Akurasi   : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-macro  : {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred, labels=labels, digits=4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=labels), display_labels=labels).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion Matrix (jumlah)")
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=labels, normalize="true"), display_labels=labels).plot(ax=ax[1], cmap="Blues", colorbar=False, values_format=".2f")
ax[1].set_title("Confusion Matrix (dinormalisasi per kelas asli)")
plt.tight_layout(); plt.show()

### Contoh kesalahan prediksi

In [ ]:
df_test = pd.DataFrame({"komentar_bersih": X_test.values, "asli": y_test.values, "prediksi": y_pred})
df_test["komentar_asli"] = df.loc[X_test.index, "comment"].values
df_test[df_test["asli"] != df_test["prediksi"]][["komentar_asli", "asli", "prediksi"]].sample(10, random_state=RANDOM_STATE)

### Kata paling berpengaruh per kelas
Diambil dari koefisien Logistic Regression (model yang mudah ditafsirkan) pada fitur TF-IDF.

In [ ]:
lr_pipe = buat_pipeline(LogisticRegression(max_iter=1000, C=1, class_weight="balanced"))
lr_pipe.set_params(tfidf__ngram_range=(1, 2)).fit(X_train, y_train)
vocab = np.array(lr_pipe.named_steps["tfidf"].get_feature_names_out())
coef  = lr_pipe.named_steps["clf"].coef_
kelas = lr_pipe.named_steps["clf"].classes_

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
warna = {"negatif": "#C44E52", "netral": "#8C8C8C", "positif": "#55A868"}
for i, k in enumerate(kelas):
    idx = np.argsort(coef[i])[-15:]
    axes[i].barh(vocab[idx], coef[i][idx], color=warna[k])
    axes[i].set_title(f"Kata/frasa penciri: {k}")
plt.tight_layout(); plt.show()

## 11. Simpan Model & Prediksi Komentar Baru

In [ ]:
joblib.dump(best_model, "model_sentimen_kdmp.joblib")
print("Model disimpan -> model_sentimen_kdmp.joblib")

def prediksi_sentimen(komentar):
    """Terima satu string atau list string, kembalikan label sentimen."""
    if isinstance(komentar, str):
        komentar = [komentar]
    bersih = [clean_text(k) for k in komentar]
    model = joblib.load("model_sentimen_kdmp.joblib")
    return pd.DataFrame({"komentar": komentar, "sentimen_prediksi": model.predict(bersih)})

prediksi_sentimen([
    "Program bagus banget, semoga bermanfaat buat masyarakat desa 👍",
    "Cuma buang-buang anggaran, ujung-ujungnya korupsi",
    "Kapan koperasinya buka di daerah saya?",
    "Harga barangnya mahal, tidak sesuai janji"
])

## 12. Kesimpulan
- Data mentah berisi banyak duplikat, komentar kosong, dan noise (stiker, emoji, slang), sehingga pembersihan mengurangi jumlah baris cukup signifikan (lihat tabel *log* pembersihan).
- Bandingkan **F1-macro** model terhadap *baseline*: selisihnya menunjukkan seberapa besar model benar-benar belajar, bukan sekadar menebak kelas mayoritas.
- Kelas **positif** paling sedikit sehingga paling sulit diprediksi; **netral** dan **negatif** sering tertukar karena banyak komentar sarkasme/pertanyaan bernada kritik.
- Ide pengembangan: *oversampling* kelas minoritas, embedding IndoBERT / IndoBERTweet, dan validasi ulang label netral yang ambigu.